In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer

# 1. تحميل البيانات
df = pd.read_csv('/content/aviationstack_flights_cleaned.csv')

# 2. حذف أعمدة غير مجدية ومعالجة الكلمات البديلة
# حذف عمود Aircraft لعدم توفر البيانات فيه بنسبة 99%
df.drop(columns=['Aircraft'], errors='ignore', inplace=True)
# استبدال "empty" و "unknown" بـ Unknown موحدة
df['Airline'] = df['Airline'].replace(['empty', 'unknown'], 'Unknown')

# 3. توحيد أنواع البيانات (Normalization)
# تحويل رقم الرحلة إلى نص نظيف بدون كسور
df['Flight Number'] = df['Flight Number'].astype(str).str.replace('\.0$', '', regex=True)

# توحيد صيغة التواريخ لكل الأعمدة الزمنية
time_cols = ['Departure Scheduled', 'Departure Actual', 'Arrival Scheduled', 'Arrival Actual']
for col in time_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# 4. معالجة التأخيرات (Delays) - استراتيجية المركز الأول
# أ- معالجة القيم المتطرفة (Winsorization عند 5% و 95%)
lower_limit = df['Arrival Delay'].quantile(0.05)
upper_limit = df['Arrival Delay'].quantile(0.95)
df['Arrival Delay_capped'] = df['Arrival Delay'].clip(lower=lower_limit, upper=upper_limit)

lower_limit_dep = df['Departure Delay'].quantile(0.05)
upper_limit_dep = df['Departure Delay'].quantile(0.95)
df['Departure Delay_capped'] = df['Departure Delay'].clip(lower=lower_limit_dep, upper=upper_limit_dep)

# ب- التحويل الرياضي لتقليل الالتواء مع الحفاظ على السالب (Yeo-Johnson)
pt = PowerTransformer(method='yeo-johnson')
df['Arrival Delay_transformed'] = pt.fit_transform(df[['Arrival Delay_capped']])
df['Departure Delay_transformed'] = pt.fit_transform(df[['Departure Delay_capped']])

# ج- تصنيف كفاءة الرحلة (Categorical Feature)
def classify_flight(delay):
    if delay < -15: return 'Early' # وصول مبكر
    elif -15 <= delay <= 15: return 'On Time'
    else: return 'Delayed'

df['Flight_Efficiency'] = df['Arrival Delay'].apply(classify_flight)

# 5. التأكد من عدم وجود مكررات ناتجة عن الـ Codeshare
df.drop_duplicates(subset=['Flight Date', 'Flight Number', 'Departure IATA'], inplace=True)

print("Data is now clean and ready for analysis! ✅")
df.to_csv('aviationstack_flights_final_cleaned.csv', index=False)


<>:16: SyntaxWarning: invalid escape sequence '\.'
<>:16: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_9831/2188600442.py:16: SyntaxWarning: invalid escape sequence '\.'
  df['Flight Number'] = df['Flight Number'].astype(str).str.replace('\.0$', '', regex=True)


Data is now clean and ready for analysis! ✅
